# 06 - Variational Autoencoders (VAE) Introduction

We now connect variational inference to representation learning.
A **Variational Autoencoder** uses latent variables to model data and trains with the ELBO.

This notebook covers:
1. The generative story behind VAEs
2. Encoder and decoder roles
3. Reconstruction term + KL term
4. A toy 2D latent-variable example

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## Part 1: The generative model

A VAE assumes data is generated by:
1. Sample latent variable $z im p(z)$
2. Sample observation $x im p_	heta(x id z)$

Usually:
- $p(z)$ is a simple prior like $athcal{N}(0, I)$
- $p_	heta(x id z)$ is a decoder network

Inference is hard because we want $p_	heta(z id x)$, which is usually intractable.

## Part 2: The encoder-decoder idea

A VAE introduces an approximate posterior
$$q_hi(z id x)$$
called the **encoder**.

The decoder is the generative model
$$p_	heta(x id z).$$

Training maximizes the ELBO for each data point $x$:
$$
athcal{L}(x) = athbb{E}_{q_hi(z id x)}[og p_	heta(x id z)] - athrm{KL}(q_hi(z id x)  p(z))
$$

This has a clear interpretation:
- first term: reconstruct the data well
- second term: keep the latent representation close to the prior

In [ ]:
# Toy latent space and decoder mapping
z = np.random.normal(0, 1, size=(400, 2))

def toy_decoder(z):
    x1 = 1.5 * z[:, 0] + 0.5 * z[:, 1]
    x2 = np.sin(z[:, 0]) + 0.2 * z[:, 1]**2
    return np.column_stack([x1, x2])

x = toy_decoder(z) + 0.1 * np.random.normal(size=(400, 2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(z[:, 0], z[:, 1], alpha=0.6, s=20, color='navy')
axes[0].set_title('Latent samples z ~ N(0, I)')
axes[0].set_xlabel('z1')
axes[0].set_ylabel('z2')

axes[1].scatter(x[:, 0], x[:, 1], alpha=0.6, s=20, color='darkorange')
axes[1].set_title('Observed data x = decoder(z) + noise')
axes[1].set_xlabel('x1')
axes[1].set_ylabel('x2')

plt.tight_layout()
plt.show()

## Part 3: The two ELBO terms

For a single data point, the ELBO is
$$
nderbrace{athbb{E}_{q_hi(z id x)}[og p_	heta(x id z)]}_{	ext{reconstruction term}} - nderbrace{athrm{KL}(q_hi(z id x)  p(z))}_{	ext{regularization term}}
$$

The reconstruction term wants the decoder to explain the data.
The KL term prevents each latent code from drifting too far away from the prior.

In [ ]:
def gaussian_kl_to_standard_normal(mu, log_var):
    return 0.5 * np.sum(np.exp(log_var) + mu**2 - 1 - log_var, axis=-1)

mu_examples = np.array([[0.0, 0.0], [1.0, -0.5], [2.0, 1.5]])
log_var_examples = np.log(np.array([[1.0, 1.0], [0.6, 1.4], [0.2, 0.3]]))
kl_vals = gaussian_kl_to_standard_normal(mu_examples, log_var_examples)

for i, kl in enumerate(kl_vals, start=1):
    print(f'Example {i}: KL to N(0, I) = {kl:.3f}')

In [ ]:
# Simple illustration of reconstruction quality
x_true = np.array([1.0, 0.3])
decoder_good = np.array([0.95, 0.25])
decoder_bad = np.array([0.1, 1.4])

recon_error_good = np.sum((x_true - decoder_good)**2)
recon_error_bad = np.sum((x_true - decoder_bad)**2)

labels = ['good reconstruction', 'bad reconstruction']
values = [recon_error_good, recon_error_bad]
plt.figure(figsize=(6, 4))
plt.bar(labels, values, color=['seagreen', 'crimson'], edgecolor='black')
plt.ylabel('squared reconstruction error')
plt.title('Reconstruction term prefers accurate decoding')
plt.show()

print('Smaller reconstruction error means larger reconstruction reward.')

## Part 4: Why reparameterization matters here

If the encoder outputs parameters $(u_hi(x), igma_hi(x))$, then the VAE samples latent codes as
$$
z = u_hi(x) + igma_hi(x) dot psilon, quad psilon im athcal{N}(0, I)
$$
which is exactly the reparameterization trick from the previous notebook.

That is what makes gradient-based training practical.

In [ ]:
# Simulate encoder outputs for a few data points
mu = np.array([[0.2, -0.1], [1.0, 0.4], [-0.5, 0.8]])
sigma = np.array([[0.8, 1.1], [0.4, 0.6], [1.2, 0.7]])
eps = np.random.normal(size=mu.shape)
z_samples = mu + sigma * eps

print('Encoder means:')
print(mu)
print('\nSampled latent codes using reparameterization:')
print(z_samples)

## Summary

What to remember:
1. A VAE is a latent-variable model trained with the ELBO
2. The encoder approximates the posterior $q_hi(zid x)$
3. The decoder defines the likelihood $p_	heta(xid z)$
4. The ELBO balances reconstruction quality and latent-space regularity
5. Reparameterization makes end-to-end optimization possible

In [ ]:
# Exercises
# 1) Change the toy decoder mapping and see how the data space changes.
# 2) Try encoder means farther from zero and inspect the KL term.
# 3) Think about what happens if the KL term is weighted too strongly or too weakly.

pass